# Dataset

In [3]:
text = ""
with open("tinyshakesphere.txt") as f:
    text = f.read()

print(text[:100])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


In [4]:
vocab = list(set(text))
vocab_size = len(vocab)
print(vocab_size)

65


In [5]:
encode = {val:i for i,val in enumerate(vocab)}
decode = {i:val for i,val in enumerate(vocab)}

print(decode[0])

Z


In [6]:
import torch
tokenized_text = torch.tensor([encode[c] for c in text],dtype=torch.long)
print(tokenized_text[:1000])

tensor([24, 17,  4, 61, 34, 32, 55, 17, 34, 17, 26, 38, 27,  1,  9, 11, 38, 40,
        15,  4, 38, 32, 18, 38, 32, 58,  4, 15, 10, 38, 38, 64, 32,  7, 27, 45,
        32, 40, 28,  4, 34, 63, 38,  4, 41, 32, 63, 38,  7,  4, 32, 46, 38, 32,
        61, 58, 38,  7, 47, 43,  9,  9, 21, 51, 51,  1,  9, 39, 58, 38,  7, 47,
        41, 32, 61, 58, 38,  7, 47, 43,  9,  9, 24, 17,  4, 61, 34, 32, 55, 17,
        34, 17, 26, 38, 27,  1,  9, 19, 15, 28, 32,  7,  4, 38, 32,  7, 51, 51,
        32,  4, 38, 61, 15, 51, 33, 38, 64, 32,  4,  7, 34, 63, 38,  4, 32, 34,
        15, 32, 64, 17, 38, 32, 34, 63,  7, 27, 32, 34, 15, 32, 40,  7, 46, 17,
        61, 63, 25,  9,  9, 21, 51, 51,  1,  9, 12, 38, 61, 15, 51, 33, 38, 64,
        43, 32,  4, 38, 61, 15, 51, 33, 38, 64, 43,  9,  9, 24, 17,  4, 61, 34,
        32, 55, 17, 34, 17, 26, 38, 27,  1,  9, 24, 17,  4, 61, 34, 41, 32, 45,
        15, 28, 32, 47, 27, 15, 18, 32, 55,  7, 17, 28, 61, 32, 56,  7,  4, 10,
        17, 28, 61, 32, 17, 61, 32, 10, 

In [7]:
train_data = tokenized_text[:int(0.9*len(text))]
val_data = tokenized_text[int(0.9*len(text)):]

print(len(train_data),len(val_data))

1003854 111540


In [8]:
block_size = 128

def get_batch(split):
    data = train_data if split == 'train' else val_data

    random_starting_indices = torch.randint(len(data) - block_size, (block_size,))

    x = torch.stack([data[i:i+block_size] for i in random_starting_indices])
    y = torch.stack([data[i+1:i+block_size+1] for i in random_starting_indices])
    return x,y
    

x,y = get_batch("train")
print(x)
print(y)


tensor([[32, 61, 38,  ..., 38, 51, 10],
        [32, 46, 38,  ..., 63, 32, 34],
        [28, 51, 64,  ..., 34, 63, 15],
        ...,
        [15,  4, 32,  ..., 27, 32, 38],
        [ 7, 10, 38,  ..., 32, 42, 38],
        [15, 18, 43,  ..., 38, 32, 63]])
tensor([[61, 38, 38,  ..., 51, 10, 15],
        [46, 38,  4,  ..., 32, 34, 63],
        [51, 64, 32,  ..., 63, 15, 28],
        ...,
        [ 4, 32,  7,  ..., 32, 38,  7],
        [10, 38, 41,  ..., 42, 38, 32],
        [18, 43,  9,  ..., 32, 63,  7]])


# Bigram Model

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class BigramModel(nn.Module):
    def __init__(self,n_embeddings, vocab_size):
        super().__init__()
        self.embedding = nn.Embedding(num_embeddings=n_embeddings, embedding_dim=vocab_size)

    def forward(self,input_encodings,targets_encodings=None):
        input_embeddings = self.embedding(input_encodings) # (B,T,C) , B = block_size, T = length of sequence, C = embedding_dim

        loss = None
        if targets_encodings is not None:
            input_embeddings_for_loss = input_embeddings.permute(0,2,1) # (B,C,T)
            loss = F.cross_entropy(input_embeddings_for_loss,targets_encodings)
        
        return loss, input_embeddings 
    
    def generate(self,input_encoding, max_output_len):
        for _ in range(max_output_len):
            _,logits = self(input_encoding)
            logits = logits[:,-1,:] # Last char of sequence
            predictions_probs = F.softmax(logits,dim=-1)
            prediction = torch.multinomial(predictions_probs, num_samples=1)
            input_encoding = torch.cat((input_encoding,prediction),dim=1)

        return input_encoding
    
model = BigramModel(vocab_size, vocab_size) # 1 row for each possible char, and for softmax, we need the vocab as output

start_context = torch.zeros((1, 1), dtype=torch.long)
gen = model.generate(start_context, 100)
print(''.join([decode[int(i)] for i in gen[0]]))



# Training Bigram

In [ ]:
device = "cuda" if torch.cuda.is_available else 'cpu'
print(device)

model = model.to(device)

In [ ]:
def train(model, n_steps, optimizer, device):
    model.train()
    
    for step in range(n_steps):
        x, y = get_batch('train')
        x, y = x.to(device), y.to(device)
        
        optimizer.zero_grad()       
        loss, _ = model(x, y)   
        loss.backward()         
        optimizer.step()            
        
        if step % 100 == 0:
            print(f"Step {step} | Train Loss = {loss.item():.4f}")


import torch.optim as optim

optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)
train(model, 4000, optimizer,device)

In [ ]:
def evaluate(model,device):
    model.eval()
    x,y = get_batch('eval')
    x, y = x.to(device), y.to(device)

    loss,_ = model(x,y)
    print("Validation Loss = ",loss.item())

evaluate(model,device)
start_context = torch.zeros((1, 1), dtype=torch.long).to(device)
gen = model.generate(start_context, 100)
print(''.join([decode[int(i)] for i in gen[0]]))


# Transformer Blackbox Model

In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class TransformerBlackbox(nn.Module):
    def __init__(self, vocab_size, d_model, nhead, num_layers, block_size):
        super().__init__()
        self.block_size = block_size
        
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.pos_embedding = nn.Embedding(block_size, d_model)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, 
            nhead=nhead, 
            batch_first=True 
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        self.lin = nn.Linear(d_model, vocab_size)

    def forward(self, input_encodings, target_encodings=None):
        B, T = input_encodings.shape

        
        tok_emb = self.token_embedding(input_encodings) # (B, T) -> (B, T, d_model)
        
        positions = torch.arange(0, T, dtype=torch.long, device=input_encodings.device)
        pos_emb = self.pos_embedding(positions) 

        x = tok_emb + pos_emb 
        
        causal_mask = nn.Transformer.generate_square_subsequent_mask(T, device=input_encodings.device)
        x = self.transformer(x, mask=causal_mask, is_causal=True)
        
        logits = self.lin(x) # (B, T, vocab_size)
        loss = None

        if target_encodings is not None:
            logits_for_loss = logits.permute(0, 2, 1)
            loss = F.cross_entropy(logits_for_loss, target_encodings)

        return loss, logits

    def generate(self, input_encoding, max_output_len):
        for _ in range(max_output_len):
            input_cond = input_encoding[:, -self.block_size:] 
            
            _, logits = self(input_cond)
            logits = logits[:, -1, :] 
            predictions_probs = F.softmax(logits, dim=-1)
            prediction = torch.multinomial(predictions_probs, num_samples=1)
            input_encoding = torch.cat((input_encoding, prediction), dim=1)

        return input_encoding

In [10]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
blackbox_model = TransformerBlackbox(vocab_size, 512, 8, 8, block_size)
blackbox_model = blackbox_model.to(device)

In [19]:
import torch.optim as optim

def train(model, n_steps, optimizer, device):
    model.train()
    
    for step in range(n_steps):
        x, y = get_batch('train')
        x, y = x.to(device), y.to(device)
        
        optimizer.zero_grad()       
        loss, _ = model(x, y)   
        loss.backward()         
        optimizer.step()            
        
        if step % 5 == 0:
            print(f"Step {step} | Train Loss = {loss.item():.4f}")

optimizer = optim.AdamW(blackbox_model.parameters(), lr=1e-4, weight_decay=0.01)
train(blackbox_model, 400, optimizer,device)

Step 0 | Train Loss = 2.3909
Step 5 | Train Loss = 2.5056
Step 10 | Train Loss = 2.4396
Step 15 | Train Loss = 2.4208
Step 20 | Train Loss = 2.4018
Step 25 | Train Loss = 2.3956
Step 30 | Train Loss = 2.4112
Step 35 | Train Loss = 2.3639
Step 40 | Train Loss = 2.3742
Step 45 | Train Loss = 2.3536
Step 50 | Train Loss = 2.3491
Step 55 | Train Loss = 2.3207
Step 60 | Train Loss = 2.3148
Step 65 | Train Loss = 2.3127
Step 70 | Train Loss = 2.2948
Step 75 | Train Loss = 2.2982
Step 80 | Train Loss = 2.2896
Step 85 | Train Loss = 2.2875
Step 90 | Train Loss = 2.2576
Step 95 | Train Loss = 2.2651
Step 100 | Train Loss = 2.2374
Step 105 | Train Loss = 2.2531
Step 110 | Train Loss = 2.2238
Step 115 | Train Loss = 2.2186
Step 120 | Train Loss = 2.2266
Step 125 | Train Loss = 2.1968
Step 130 | Train Loss = 2.1747
Step 135 | Train Loss = 2.1751
Step 140 | Train Loss = 2.1994
Step 145 | Train Loss = 2.1740
Step 150 | Train Loss = 2.1599
Step 155 | Train Loss = 2.1312
Step 160 | Train Loss = 2.1476

In [20]:
def evaluate(model,device):
    model.eval()
    x,y = get_batch('eval')
    x, y = x.to(device), y.to(device)

    loss,_ = model(x,y)
    print("Validation Loss = ",loss.item())

evaluate(blackbox_model,device)

inp_text = "Hello"
inp_text_tokenized = torch.tensor([encode[c] for c in inp_text], dtype=torch.long).unsqueeze(0).to(device)
gen = blackbox_model.generate(inp_text_tokenized, 100)
print(''.join([decode[int(i)] for i in gen[0]]))


Validation Loss =  1.899012804031372
Hellookesslice
Ofied, ploose ime? a bearmal aBut his tur?


QUEEN:
ENaw you you, thou shim; trost heart,



In [22]:

torch.save(blackbox_model.state_dict(), 'blackbox_transformer_model.pt')
print("Model successfully saved!")

Model successfully saved!
